# Chapter 2: PyTorch — Full Control, Industry Standard for New Research

### Why This Matters

PyTorch trades some of Keras's convenience for explicit, granular control over every step of training — and it has become the dominant framework for new deep learning research and much of production ML precisely because that explicitness makes complex, non-standard architectures easier to build and debug.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# 2.1 Tensors and automatic differentiation
x = torch.tensor(3.0, requires_grad=True)     # requires_grad=True: "track operations on this for gradient computation"
y = x**2 + 2*x
y.backward()                                    # compute dy/dx via backpropagation
print(x.grad)                                   # 2*x + 2 = 8.0, computed automatically

This tiny example is worth sitting with, because it's the entire conceptual foundation of the framework: `requires_grad=True` tells PyTorch to record every operation performed on `x`; `.backward()` walks that recorded computation graph backward, applying the chain rule automatically to compute the exact derivative — no calculus by hand required, no matter how deep or complex the actual network gets. This mechanism, called **autograd**, is what makes training an arbitrarily complex neural network as simple (mechanically) as this two-line example.

### 2.2 Defining a Model

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()                       # remember Volume 01's OOP chapter — this is the same pattern
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

A PyTorch model is a Python class inheriting from `nn.Module` — directly the inheritance pattern from Volume 01, Chapter 12. `__init__` defines the layers as attributes; `forward()` defines how data flows through them. This object-oriented shape, rather than Keras's more declarative `Sequential` list, is exactly why PyTorch gives you more control: `forward()` is ordinary Python code, so it can contain loops, conditionals, or any custom logic a fixed layer stack couldn't express.

### 2.3 The Training Loop — Explicit, Not Hidden

In [ ]:
model = SimpleNet(n_features)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(30):
    model.train()                                # tells layers like Dropout to behave in "training mode"
    for xb, yb in train_loader:
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()                    # clear gradients from the PREVIOUS batch — easy to forget!
        loss.backward()                          # compute new gradients
        optimizer.step()                         # update every weight using those gradients

Unlike Keras's single `.fit()` call, PyTorch makes every step of the training loop explicit — you can see exactly where the forward pass, loss calculation, and weight update each happen. This is more code to write, but it's precisely why customizing training behavior (a novel loss function, an unusual data flow, mixing multiple models) is more natural in PyTorch than fighting against a higher-level framework's assumptions.

⚠️ **`optimizer.zero_grad()` is the single most common bug for PyTorch beginners to forget.** PyTorch *accumulates* gradients by default rather than overwriting them on each `.backward()` call — if you skip `zero_grad()`, each batch's gradient calculation gets added on top of the previous batch's, silently corrupting training. Get in the habit of writing all three lines (`zero_grad → backward → step`) together, every time, as a fixed unit.

`model.train()` vs. `model.eval()` matters because some layers (`Dropout`, `BatchNorm`) behave differently during training versus inference — `Dropout` randomly zeroes neurons only during training, and should be fully "on" (using every neuron) during evaluation/prediction. Forgetting to call `model.eval()` before evaluating your model on validation or test data is a second extremely common and hard-to-notice bug — it will still run, just with subtly incorrect (usually worse, more randomly noisy) results.

### Cheat Sheet — PyTorch

| Task | Code |
|---|---|
| Track gradients | `torch.tensor(x, requires_grad=True)` |
| Compute gradients | `loss.backward()` |
| Define a model | `class Net(nn.Module): def __init__/forward` |
| Training loop unit | `zero_grad() → backward() → step()`, always together |
| Switch modes | `model.train()` / `model.eval()` |
| Save / load | `torch.save(model.state_dict(), path)`, `model.load_state_dict(torch.load(path))` |

---